In [2]:
# ============================================================
# Baseline Federated Training (NO Preprocessing)
# MobileNetV2 + Adapter + Head
# ============================================================

import os
import time
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
 
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    classification_report,
    accuracy_score
)
from sklearn.preprocessing import label_binarize

# ============================================================
# 0) CONFIG
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TRAIN_DIR = "/Users/sabbirahmed/Downloads/Rafi/train_data_no_aug"
TEST_DIR  = "/Users/sabbirahmed/Downloads/Rafi/Fish/test_data"


IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
LR = 1e-4
DROPOUT = 0.3

NUM_CLIENTS = 4
FL_ROUNDS = 10

# ============================================================
# LOGGING
# ============================================================
def ts():
    return time.strftime("%H:%M:%S")

def log(msg):
    print(f"[{ts()}] [INFO] {msg}")

def phase(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

# ============================================================
# 1) DATA LOADING
# ============================================================
phase("PHASE 0: Load dataset and split into clients")

def list_classes(train_dir):
    classes = sorted([
        d for d in os.listdir(train_dir)
        if os.path.isdir(os.path.join(train_dir, d))
    ])
    return classes, {c: i for i, c in enumerate(classes)}

def load_paths_labels(data_dir, class_to_idx):
    paths, labels = [], []
    for cls, idx in class_to_idx.items():
        cls_path = os.path.join(data_dir, cls)
        for fn in os.listdir(cls_path):
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(cls_path, fn))
                labels.append(idx)
    return np.array(paths), np.array(labels)

def split_clients(paths, labels, n=4):
    idx = np.random.permutation(len(paths))
    paths, labels = paths[idx], labels[idx]

    sizes = (np.random.dirichlet([1]*n) * len(paths)).astype(int)
    sizes[-1] = len(paths) - sum(sizes[:-1])

    out, s = [], 0
    for i, sz in enumerate(sizes):
        out.append((paths[s:s+sz], labels[s:s+sz]))
        log(f"Client C{i+1}: {sz} samples")
        s += sz
    return out

class_names, class_to_idx = list_classes(TRAIN_DIR)
K = len(class_names)
log(f"Detected {K} classes")

train_paths, train_labels = load_paths_labels(TRAIN_DIR, class_to_idx)
test_paths,  test_labels  = load_paths_labels(TEST_DIR,  class_to_idx)

clients = split_clients(train_paths, train_labels, NUM_CLIENTS)

# ============================================================
# 2) IMAGE DECODING (NO PREPROCESSING)
# ============================================================
phase("PHASE 1: Image decoding (resize + normalize only)")

def decode_and_resize(path_tensor):
    raw = tf.io.read_file(path_tensor)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def make_client_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(
            buffer_size=min(2000, len(paths)),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    def map_fn(x, y):
        img = decode_and_resize(x)
        return img, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# ============================================================
# 3) MODEL (SAME AS YOUR BASE MODEL)
# ============================================================
phase("PHASE 0 (Model): Build MobileNetV2 + adapter + head")

def build_model(num_classes):
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
    )
    base.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dense(128, activation="relu", name="adapter")(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    out = tf.keras.layers.Dense(num_classes, activation="softmax", name="head")(x)

    return tf.keras.Model(base.input, out)

global_model = build_model(K)
log("Model built. Backbone frozen. Adapter + head trainable.")

# ============================================================
# 4) FEDERATED TRAINING (FedAvg)
# ============================================================
phase("PHASE 2: Federated training (NO preprocessing)")

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

def get_trainable_vars(model):
    return model.get_layer("adapter").weights + model.get_layer("head").weights

def extract_trainable_numpy(model):
    return [v.numpy() for v in get_trainable_vars(model)]

def assign_trainable_numpy(model, vals):
    for var, val in zip(get_trainable_vars(model), vals):
        var.assign(val)

def fedavg(weight_sets, sizes):
    total = float(sum(sizes))
    avg = []
    for wi in range(len(weight_sets[0])):
        acc = 0.0
        for ci in range(len(weight_sets)):
            acc += (sizes[ci] / total) * weight_sets[ci][wi]
        avg.append(acc)
    return avg

def train_one_epoch(model, dataset, optimizer, client_id, round_id):
    total_loss = 0.0
    steps = 0
    for xb, yb in dataset:
        with tf.GradientTape() as tape:
            preds = model(xb, training=True)
            loss = loss_fn(yb, preds)
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        total_loss += float(loss.numpy())
        steps += 1
    avg_loss = total_loss / max(steps, 1)
    log(f"[Round {round_id}] Client C{client_id} avg loss = {avg_loss:.4f}")
    return avg_loss

# ---------- FL LOOP ----------
for r in range(1, FL_ROUNDS + 1):
    phase(f"FL ROUND {r}/{FL_ROUNDS}")

    global_weights = global_model.get_weights()
    local_weights = []
    sizes = []

    for ci, (cp, cl) in enumerate(clients, start=1):
        log(f"Client C{ci}: sync global model")
        local_model = build_model(K)
        local_model.set_weights(global_weights)

        ds = make_client_dataset(cp, cl, shuffle=True)
        opt = tf.keras.optimizers.Adam(LR)

        train_one_epoch(local_model, ds, opt, ci, r)

        local_weights.append(extract_trainable_numpy(local_model))
        sizes.append(len(cp))

    log("Server: FedAvg aggregation")
    avg_weights = fedavg(local_weights, sizes)
    assign_trainable_numpy(global_model, avg_weights)
    log("Server: global model updated")

# ============================================================
# 5) FINAL TEST EVALUATION (ALL METRICS)
# ============================================================
phase("FINAL EVALUATION: Test set (NO preprocessing)")

def make_test_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def map_fn(x, y):
        img = decode_and_resize(x)
        return img, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

# ---------------------------
# Prediction
# ---------------------------
test_ds = make_test_dataset(test_paths, test_labels)
pred_probs = global_model.predict(test_ds, verbose=1)
pred_labels = np.argmax(pred_probs, axis=1)

# ---------------------------
# Basic Metrics
# ---------------------------
acc  = accuracy_score(test_labels, pred_labels)
prec = precision_score(test_labels, pred_labels, average="weighted", zero_division=0)
rec  = recall_score(test_labels, pred_labels, average="weighted", zero_division=0)
f1   = f1_score(test_labels, pred_labels, average="weighted", zero_division=0)
ll   = log_loss(test_labels, pred_probs)

# ---------------------------
# ROC-AUC & PR-AUC (Multiclass OvR)
# ---------------------------
y_true_bin = label_binarize(test_labels, classes=list(range(K)))

roc_auc = roc_auc_score(
    y_true_bin,
    pred_probs,
    average="macro",
    multi_class="ovr"
)

pr_auc = average_precision_score(
    y_true_bin,
    pred_probs,
    average="macro"
)

# ---------------------------
# Logging Results
# ---------------------------
log(f"Accuracy     : {acc:.4f}")
log(f"Precision    : {prec:.4f}")
log(f"Recall       : {rec:.4f}")
log(f"F1-Score     : {f1:.4f}")
log(f"ROC-AUC      : {roc_auc:.4f}")
log(f"PR-AUC       : {pr_auc:.4f}")
log(f"Log-Loss     : {ll:.4f}")

# ---------------------------
# Detailed Report
# ---------------------------
print("\nClassification Report:")
print(classification_report(
    test_labels,
    pred_labels,
    target_names=class_names,
    digits=4
))


from sklearn.metrics import matthews_corrcoef

mcc = matthews_corrcoef(test_labels, pred_labels)
log(f"MCC Score    = {mcc:.4f}")



def visualize_before_after(paths, pipeline, n=8, title="Before / After"):
    sel = np.random.choice(len(paths), size=n, replace=False)
    fig, axes = plt.subplots(n, 2, figsize=(6, 3*n))

    for i, idx in enumerate(sel):
        img = decode_and_resize(tf.constant(paths[idx])).numpy()
        proc = tf_apply_pipeline(tf.constant(img), pipeline).numpy()

        axes[i,0].imshow(img)
        axes[i,0].set_title("Original")
        axes[i,0].axis("off")

        axes[i,1].imshow(proc)
        axes[i,1].set_title("Processed")
        axes[i,1].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

visualize_before_after(
    clients[0][0],
    client_pipelines[0],
    title="Client C1 Preprocessing Validation"
)

def visualize_difference_map(paths, pipeline, n=6):
    sel = np.random.choice(len(paths), size=n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3*n))

    for i, idx in enumerate(sel):
        img = decode_and_resize(tf.constant(paths[idx])).numpy()
        proc = tf_apply_pipeline(tf.constant(img), pipeline).numpy()
        diff = np.abs(proc - img)

        axes[i,0].imshow(img)
        axes[i,0].set_title("Original")
        axes[i,0].axis("off")

        axes[i,1].imshow(proc)
        axes[i,1].set_title("Processed")
        axes[i,1].axis("off")

        axes[i,2].imshow(diff, cmap="hot")
        axes[i,2].set_title("Difference Map")
        axes[i,2].axis("off")

    plt.tight_layout()
    plt.show()

visualize_difference_map(
    clients[0][0],
    client_pipelines[0]
)


def compare_multiple_ga_runs(client_id, cp, cl, runs=3):
    pipes = []
    for r in range(runs):
        log(f"GA consistency run {r+1}/{runs} for Client C{client_id}")
        pipe = evolve_pipeline_for_client(client_id, cp, cl)
        pipes.append(pipe)

    for i, p in enumerate(pipes):
        log(f"Run {i+1} pipeline: {pretty_pipeline(p)}")

    return pipes

compare_multiple_ga_runs(
    client_id=1,
    cp=clients[0][0],
    cl=clients[0][1],
    runs=3
)



PHASE 0: Load dataset and split into clients
[10:15:15] [INFO] Detected 23 classes
[10:15:15] [INFO] Client C1: 258 samples
[10:15:15] [INFO] Client C2: 1722 samples
[10:15:15] [INFO] Client C3: 2146 samples
[10:15:15] [INFO] Client C4: 279 samples

PHASE 1: Image decoding (resize + normalize only)

PHASE 0 (Model): Build MobileNetV2 + adapter + head
[10:15:16] [INFO] Model built. Backbone frozen. Adapter + head trainable.

PHASE 2: Federated training (NO preprocessing)

FL ROUND 1/10
[10:15:16] [INFO] Client C1: sync global model
[10:15:45] [INFO] [Round 1] Client C1 avg loss = 3.2278
[10:15:45] [INFO] Client C2: sync global model
[10:18:23] [INFO] [Round 1] Client C2 avg loss = 2.5373
[10:18:23] [INFO] Client C3: sync global model
[10:20:45] [INFO] [Round 1] Client C3 avg loss = 2.3617
[10:20:45] [INFO] Client C4: sync global model
[10:21:03] [INFO] [Round 1] Client C4 avg loss = 3.3951
[10:21:03] [INFO] Server: FedAvg aggregation
[10:21:03] [INFO] Server: global model updated

FL R

2026-01-01 10:43:30.177332: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[10:43:30] [INFO] [Round 6] Client C4 avg loss = 0.4716
[10:43:30] [INFO] Server: FedAvg aggregation
[10:43:30] [INFO] Server: global model updated

FL ROUND 7/10
[10:43:30] [INFO] Client C1: sync global model
[10:43:47] [INFO] [Round 7] Client C1 avg loss = 0.4133
[10:43:47] [INFO] Client C2: sync global model
[10:45:39] [INFO] [Round 7] Client C2 avg loss = 0.3737
[10:45:39] [INFO] Client C3: sync global model
[10:47:57] [INFO] [Round 7] Client C3 avg loss = 0.3352
[10:47:57] [INFO] Client C4: sync global model
[10:48:16] [INFO] [Round 7] Client C4 avg loss = 0.4277
[10:48:16] [INFO] Server: FedAvg aggregation
[10:48:16] [INFO] Server: global model updated

FL ROUND 8/10
[10:48:16] [INFO] Client C1: sync global model
[10:48:34] [INFO] [Round 8] Client C1 avg loss = 0.3704
[10:48:34] [INFO] Client C2: sync global model
[10:50:24] [INFO] [Round 8] Client C2 avg loss = 0.3244
[10:50:24] [INFO] Client C3: sync global model
[10:52:44] [INFO] [Round 8] Client C3 avg loss = 0.2743
[10:52:44

NameError: name 'client_pipelines' is not defined